In [3]:
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

SEED = 42

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Divisão estratificada e reproduzível: 60% treino, 20% validação, 20% teste (cofre)
df_resto, df_teste = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df["Churn"]
)
df_treino, df_valid = train_test_split(
    df_resto, test_size=0.25, random_state=SEED, stratify=df_resto["Churn"]
)

print(f"Total: {len(df)} | Treino: {len(df_treino)} | Validação: {len(df_valid)} | Teste (cofre): {len(df_teste)}")

Total: 7043 | Treino: 4225 | Validação: 1409 | Teste (cofre): 1409


In [4]:
def limpar_dados(dados):
    d = dados.copy()
    d["TotalCharges"] = pd.to_numeric(d["TotalCharges"], errors="coerce").fillna(0.0)
    d["Churn_num"] = d["Churn"].map({"No": 0, "Yes": 1})
    return d

df_treino = limpar_dados(df_treino)
df_valid = limpar_dados(df_valid)
df_teste = limpar_dados(df_teste)

# Excluir customerID e alvo das entradas (X) e separar alvo 0/1 (y)
X_treino = df_treino.drop(columns=["customerID", "Churn", "Churn_num"])
y_treino = df_treino["Churn_num"].values

X_valid = df_valid.drop(columns=["customerID", "Churn", "Churn_num"])
y_valid = df_valid["Churn_num"].values

X_teste = df_teste.drop(columns=["customerID", "Churn", "Churn_num"])
y_teste = df_teste["Churn_num"].values

print(f"Proporção Churn -> Treino: {y_treino.mean():.4f} | Validação: {y_valid.mean():.4f}")

Proporção Churn -> Treino: 0.2653 | Validação: 0.2654


In [5]:
colunas_numericas = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
colunas_categoricas = [c for c in X_treino.columns if c not in colunas_numericas]

preprocessador = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), colunas_numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), colunas_categoricas),
    ]
)

# Ajustar (fit) exclusivamente no conjunto de treino
X_treino_proc = preprocessador.fit_transform(X_treino)

# Apenas transformar (transform) validação e teste
X_valid_proc = preprocessador.transform(X_valid)
X_teste_proc = preprocessador.transform(X_teste) # Reservado para a avaliação final do tech lead

# Salvar para uso no baseline, Keras e função de previsão
joblib.dump(preprocessador, "preprocessador.joblib")

print(f"X_treino_proc: {X_treino_proc.shape} | X_valid_proc: {X_valid_proc.shape}")
print("Pré-processador salvo em: preprocessador.joblib")

X_treino_proc: (4225, 45) | X_valid_proc: (1409, 45)
Pré-processador salvo em: preprocessador.joblib


In [6]:
schema_campos = {}

for col in X_treino.columns:
    if col in colunas_numericas:
        schema_campos[col] = {
            "tipo": "int" if col in ["SeniorCitizen", "tenure"] else "float",
            "min": float(X_treino[col].min()),
            "max": float(X_treino[col].max()),
            "mediana_padrao": float(X_treino[col].median())
        }
    else:
        schema_campos[col] = {
            "tipo": "categoria",
            "valores_aceitos": sorted(X_treino[col].unique().tolist()),
            "valor_padrao": X_treino[col].mode()[0]
        }

with open("schema_campos.json", "w", encoding="utf-8") as f:
    json.dump(schema_campos, f, indent=2, ensure_ascii=False)

print("Schema salvo em: schema_campos.json\n")
for campo, info in schema_campos.items():
    if info["tipo"] == "categoria":
        print(f"- {campo} ({info['tipo']}): {info['valores_aceitos']}")
    else:
        print(f"- {campo} ({info['tipo']}): min={info['min']}, max={info['max']}")

Schema salvo em: schema_campos.json

- gender (categoria): ['Female', 'Male']
- SeniorCitizen (int): min=0.0, max=1.0
- Partner (categoria): ['No', 'Yes']
- Dependents (categoria): ['No', 'Yes']
- tenure (int): min=0.0, max=72.0
- PhoneService (categoria): ['No', 'Yes']
- MultipleLines (categoria): ['No', 'No phone service', 'Yes']
- InternetService (categoria): ['DSL', 'Fiber optic', 'No']
- OnlineSecurity (categoria): ['No', 'No internet service', 'Yes']
- OnlineBackup (categoria): ['No', 'No internet service', 'Yes']
- DeviceProtection (categoria): ['No', 'No internet service', 'Yes']
- TechSupport (categoria): ['No', 'No internet service', 'Yes']
- StreamingTV (categoria): ['No', 'No internet service', 'Yes']
- StreamingMovies (categoria): ['No', 'No internet service', 'Yes']
- Contract (categoria): ['Month-to-month', 'One year', 'Two year']
- PaperlessBilling (categoria): ['No', 'Yes']
- PaymentMethod (categoria): ['Bank transfer (automatic)', 'Credit card (automatic)', 'Electroni